# 9주차 과제 정답지

In [ ]:
!pip install -q langchain langchain-core langchain-openai langchain-google-genai langgraph tavily-python python-dotenv

In [ ]:

# 로컬 환경: 프로젝트 루트의 .env 파일에 키를 저장하고 아래 코드를 사용
# from dotenv import load_dotenv
# load_dotenv(override=True)

# Colab 환경: 위의 로컬 환경 설정을 주석 처리하고 아래 코드를 활성화
# API 키 설정
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

True

## 문제 1 정답. LangChain 기본 모델 호출

In [2]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

model = init_chat_model(model="openai:gpt-4o-mini")

response = model.invoke([HumanMessage(content="LangChain이 무엇인지 한 문장으로 설명해줘.")])

print(response.content)

LangChain은 언어 모델을 활용하여 다양한 작업을 자동화하고 응용 프로그램을 구축하기 위한 프레임워크입니다.


---
## 문제 2 정답. SystemMessage + HumanMessage 대화 구성

In [10]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="너는 AI 에이전트 전문가야. 모든 답변을 2문장 이내로 해줘."),
    HumanMessage(content="LangGraph를 설명해줘."),
]

response = model.invoke(messages)

print(response.content)

LangGraph는 자연어 처리와 그래프 구조를 결합하여 정보를 효율적으로 표현하고 검색하는 기술입니다. 이 시스템은 언어적 관계를 시각적으로 모델링해 데이터 간의 연결성을 강조합니다.


---
## 문제 3 정답. Structured Output

In [11]:
from pydantic import BaseModel, Field

class FrameworkInfo(BaseModel):
    name: str = Field(description="AI 프레임워크 이름")
    description: str = Field(description="한 줄 설명")

structured_model = model.with_structured_output(FrameworkInfo)

result = structured_model.invoke([HumanMessage(content="LangGraph를 소개해줘.")])

print(f"name: {result.name}")
print(f"description: {result.description}")

name: LangGraph
description: LangGraph는 다양한 언어 처리 작업을 위한 그래프 기반의 구조를 제공하는 AI 프레임워크입니다. 이 프레임워크는 자연어 처리(NLP) 및 그래프 신경망(GNN)의 결합을 통해 텍스트 데이터를 이해하고 분석하는 데 강력한 도구로 사용됩니다. LangGraph는 언어 모델의 성능을 개선하고 문맥을 보다 깊이 이해할 수 있도록 설계되어 있으며, 여러 언어의 데이터를 처리하는 데 적합합니다.


---
## 문제 4 정답. LangGraph State 정의

In [14]:
import operator
from typing_extensions import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    topic: str
    summary: str

assert "messages" in AgentState.__annotations__
assert "topic" in AgentState.__annotations__
assert "summary" in AgentState.__annotations__
print("State 정의 완료:", list(AgentState.__annotations__.keys()))

State 정의 완료: ['messages', 'topic', 'summary']


---
## 문제 5 정답. StateGraph 기본 구성

In [18]:
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END

def respond(state: AgentState):
    last_content = state["messages"][-1].content
    return {"messages": [AIMessage(content=f"{last_content} 에 대해 답변드릴게요.")]}

builder = StateGraph(AgentState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)

graph = builder.compile()
result = graph.invoke({"messages": [HumanMessage(content="LangGraph")]})
print(result["messages"][-1].content)

LangGraph 에 대해 답변드릴게요.


---
## 문제 6 정답. 조건부 라우팅

In [17]:
from typing_extensions import Literal

def route(state: AgentState) -> Literal["long", "short"]:
    last_content = state["messages"][-1].content
    if len(last_content) >= 10:
        return "long"
    return "short"

def long_response(state: AgentState):
    return {"messages": [AIMessage(content="긴 질문이에요.")]}

def short_response(state: AgentState):
    return {"messages": [AIMessage(content="짧은 질문이에요.")]}

builder = StateGraph(AgentState)
builder.add_node("long", long_response)
builder.add_node("short", short_response)
builder.add_conditional_edges(
    START,
    route,
    {"long": "long", "short": "short"}
)
builder.add_edge("long", END)
builder.add_edge("short", END)

graph = builder.compile()

result_long = graph.invoke({"messages": [HumanMessage(content="LangGraph에 대해 자세히 설명해줘")]})
result_short = graph.invoke({"messages": [HumanMessage(content="안녕")]})
print(result_long["messages"][-1].content)
print(result_short["messages"][-1].content)

긴 질문이에요.
짧은 질문이에요.


---
## 문제 7 정답. Tool 정의 및 tool_node 구현

In [27]:
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import ToolMessage, AIMessage
from typing_extensions import Annotated

@tool(parse_docstring=True)
def word_count_tool(
    text: str,
) -> str:
    """텍스트의 단어 수를 반환한다.

    Args:
        text: 단어 수를 셀 텍스트
    """
    count = len(text.split())
    return str(count)

tools = [word_count_tool]
tools_by_name = {tool.name: tool for tool in tools}

def tool_node(state):
    tool_calls = state["messages"][-1].tool_calls
    outputs = []
    for tc in tool_calls:
        tool = tools_by_name[tc["name"]]
        observation = tool.invoke(tc)
        outputs.append(observation)
    return {"messages": outputs}

fake_ai_msg = AIMessage(
    content="",
    tool_calls=[{"name": "word_count_tool", "args": {"text": "Hello LangGrpah"}, "id": "tc1", "type": "tool_call"}]
)
test_result = tool_node({"messages": [fake_ai_msg]})
assert len(test_result["messages"]) == 1
assert isinstance(test_result["messages"][0], ToolMessage)
print("tool_node 테스트 통과")
print("단어 수 결과:", test_result["messages"][0].content)

tool_node 테스트 통과
단어 수 결과: 2


---
## 문제 8 정답. output_schema 적용 및 실행 검증

In [28]:
class ResearchState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    result: str

class ResearchOutputState(TypedDict):
    result: str

def llm_call(state: ResearchState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

def finalize(state: ResearchState):
    last_content = state["messages"][-1].content
    return {"result": last_content}

research_builder = StateGraph(ResearchState, output_schema=ResearchOutputState)
research_builder.add_node("llm_call", llm_call)
research_builder.add_node("finalize", finalize)
research_builder.add_edge(START, "llm_call")
research_builder.add_edge("llm_call", "finalize")
research_builder.add_edge("finalize", END)

research_graph = research_builder.compile()

result = research_graph.invoke({"messages": [HumanMessage(content="LangGraph를 한 문장으로 설명해줘.")]})

print("result keys:", result.keys())
assert set(result.keys()) == {"result"}, f"output_schema가 올바르지 않습니다: {result.keys()}"
print("output_schema 검증 통과")
print("result:", result["result"])

result keys: dict_keys(['result'])
✅ output_schema 검증 통과
result: LangGraph는 다양한 언어의 구조와 의미를 시각적으로 표현하여 언어 간의 관계를 이해하고 분석하는 도구입니다.


---
## 문제 9 정답. ReAct 에이전트 전체 구현

In [31]:
from tavily import TavilyClient
from langchain_core.messages import SystemMessage, ToolMessage

tavily_client = TavilyClient()

@tool(parse_docstring=True)
def simple_search(query: str) -> str:
    """웹을 검색하고 결과를 반환한다.

    Args:
        query: 검색 쿼리
    """
    result = tavily_client.search(query, max_results=2)
    return str(result["results"])

class ReactState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

react_model = init_chat_model(model="openai:gpt-4o-mini")
react_model_with_tools = react_model.bind_tools([simple_search])

def react_llm_call(state: ReactState):
    response = react_model_with_tools.invoke(
        [SystemMessage(content="너는 웹 검색 에이전트야.")] + state["messages"]
    )
    return {"messages": [response]}

react_tools_by_name = {simple_search.name: simple_search}

def react_tool_node(state: ReactState):
    tool_calls = state["messages"][-1].tool_calls
    outputs = []
    for tc in tool_calls:
        obs = react_tools_by_name[tc["name"]].invoke(tc["args"])
        outputs.append(ToolMessage(content=str(obs), name=tc["name"], tool_call_id=tc["id"]))
    return {"messages": outputs}

def should_continue(state: ReactState) -> Literal["tool_node", "__end__"]:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tool_node"
    return "__end__"

react_builder = StateGraph(ReactState)
react_builder.add_node("llm_call", react_llm_call)
react_builder.add_node("tool_node", react_tool_node)
react_builder.add_edge(START, "llm_call")
react_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    {"tool_node": "tool_node", "__end__": END}
)
react_builder.add_edge("tool_node", "llm_call")

react_graph = react_builder.compile()

result = react_graph.invoke(
    {"messages": [HumanMessage(content="2026년 LangChain의 최신 버전을 검색해줘.")]}
)
print(result["messages"][-1].content)

2026년의 LangChain 최신 버전은 다음과 같습니다:

1. **PyPI**: 최신 버전은 2026년 4월 3일에 출시된 것으로, [여기](https://pypi.org/project/langchain/)서 확인할 수 있습니다.

2. **GitHub**: 최신 릴리스는 2026년 4월 10일에 발표된 langchain-core==1.3.0a1입니다. 더 많은 정보는 [여기](https://github.com/langchain-ai/langchain/releases)에서 확인할 수 있습니다. 

추가적인 정보를 원하시면 해당 링크들을 방문해 주세요.


---
## 문제 10 정답. 멀티턴 대화 + compress 노드 구현

In [33]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import filter_messages

class MultiTurnState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    compressed_result: str

def research_node(state: MultiTurnState):
    research_model = init_chat_model(model="openai:gpt-4o-mini")
    response = research_model.invoke(state["messages"])
    return {"messages": [response]}

compress_model = init_chat_model(model="google_genai:gemini-2.5-flash")

def compress_node(state: MultiTurnState):
    ai_messages = filter_messages(state["messages"], include_types=["ai"])
    combined = "\n".join([m.content for m in ai_messages])
    response = compress_model.invoke(
        [HumanMessage(content=f"다음 내용을 한 문장으로 요약해줘:\n{combined}")]
    )
    return {"compressed_result": response.content}

checkpointer = InMemorySaver()

mt_builder = StateGraph(MultiTurnState)
mt_builder.add_node("research_node", research_node)
mt_builder.add_node("compress_node", compress_node)
mt_builder.add_edge(START, "research_node")
mt_builder.add_edge("research_node", "compress_node")
mt_builder.add_edge("compress_node", END)

mt_graph = mt_builder.compile(checkpointer=checkpointer)

thread = {"configurable": {"thread_id": "turn-test"}}

result1 = mt_graph.invoke(
    {"messages": [HumanMessage(content="LangGraph의 주요 특징을 알려줘.")]},
    config=thread
)
print(f"Turn 1 messages: {len(result1['messages'])}개")

result2 = mt_graph.invoke(
    {"messages": [HumanMessage(content="그 중 가장 중요한 것은 뭐야?")]},
    config=thread
)
print(f"Turn 2 messages: {len(result2['messages'])}개")

assert len(result2["messages"]) > len(result1["messages"]), \
    "멀티턴 대화에서 messages가 누적되지 않았습니다."
print("멀티턴 누적 확인")
print("Turn 2 compressed:", result2["compressed_result"])

Turn 1 messages: 2개
Turn 2 messages: 4개
멀티턴 누적 확인
Turn 2 compressed: LangGraph는 자연어 처리(NLP)와 그래프 이론을 결합한 프레임워크로, **그래프 구조**를 활용하여 텍스트 데이터 간의 복잡한 관계를 시각화하고 분석함으로써 의미론적 이해를 심화하는 데 핵심적인 역할을 합니다.
